# FirstSlip Phase 4: training E4 (SFT and GRPO) on Colab

Trains Qwen3.5-2B with LoRA on the synthetic train split, in two ways: **SFT** on the step-only target and **GRPO** with the SymPy checker as reward. Three seeds each. All the logic is in `scripts/phase4_train.py`, `scripts/phase4_eval.py` and `config/phase4.json`; this notebook only sets up the environment and calls them.

- Runtime: **Colab Pro, A100**. Keep each GRPO run **under 3 hours**.
- Adapters and run logs are copied to Google Drive after every step, so a disconnect loses at most one run.
- **Before Step 2:** read C3 (DeepSeekMath, GRPO section) and C4 (TRL GRPOTrainer, custom reward section).
- At the end, download `phase4_runs.zip` and unzip it into the repo's `runs/` folder, then run `python scripts/phase4_score.py` locally.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone https://github.com/AbdulRehmanAntall/firstslip.git
%cd firstslip
!git log -1 --format='%H %s'

In [ ]:
# Repo stack first, then TRL with its vLLM extra, so TRL's pins win.
!pip install -q -r requirements.txt
!pip install -q "trl[vllm]==1.16.0" peft matplotlib
import importlib.metadata as md
print({p: md.version(p) for p in ("trl", "peft", "vllm", "torch", "transformers", "datasets")})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/firstslip_phase4'
CKPT = f'{DRIVE}/checkpoints'
!mkdir -p {CKPT} {DRIVE}/runs
def save_runs():
    !cp -n runs/*_phase4*.jsonl {DRIVE}/runs/ 2>/dev/null; ls {DRIVE}/runs | wc -l

Regenerate the synthetic data and the Phase 4 training rows (deterministic, about 10 minutes of SymPy) and check both against the committed hashes.

In [ ]:
!python scripts/generate_synthetic.py > /dev/null
!python scripts/phase4_prepare.py
import hashlib, json, pathlib
for d in ('synthetic', 'phase4'):
    man = json.loads(pathlib.Path(f'data/{d}/manifest.json').read_text())
    for split, meta in man['splits'].items():
        h = hashlib.sha256(pathlib.Path(f'data/{d}/{split}.jsonl').read_bytes()).hexdigest()
        assert h == meta['sha256'], f'{d}/{split}: hash mismatch'
        print(d, split, 'OK')

## Step 1: smoke tests (5 steps each)

Checks that the model loads, that the SFT target is exactly `Step: k
Feedback:`, that GRPO generates through vLLM and gets rewards, and that an adapter can be merged and evaluated. If Qwen3.5-2B fails here, add `--fallback` (Qwen3-1.7B) to **every** command below, including the base eval, and record the switch in PROGRESS.md.

In [ ]:
FB = ''  # set to '--fallback' to use Qwen3-1.7B
!python scripts/phase4_train.py sft --seed 0 --max-steps 5 --out {CKPT} {FB}
!python scripts/phase4_train.py grpo --seed 0 --max-steps 5 --out {CKPT} {FB}
MODEL = 'qwen3-1.7b' if FB else 'qwen-2b'
!python scripts/phase4_eval.py --label smoke-grpo --adapter {CKPT}/{MODEL}/grpo-s0-smoke5/final --splits dev --sample 40 {FB}

## Step 2: base model and SFT (3 seeds)

The base model is greedy, so it has one run. Each SFT run is one epoch over 8,000 items.

In [ ]:
!python scripts/phase4_eval.py --label base {FB}
save_runs()
for s in (0, 1, 2):
    !python scripts/phase4_train.py sft --seed {s} --out {CKPT} {FB}
    !python scripts/phase4_eval.py --label sft-s{s} --adapter {CKPT}/{MODEL}/sft-s{s}/final {FB}
    save_runs()

## Step 3: GRPO seed 0 and the go/no-go (12 Nov)

Watch `reward` and `frac_reward_zero_std` in the log. Then score each saved checkpoint on dev.

**Go/no-go:** if mean reward has not risen **and** dev accuracy is not above base, switch to rejection-sampling fine-tuning (ask Claude to add the `rft` method) instead of running Step 4.

In [ ]:
!python scripts/phase4_train.py grpo --seed 0 --out {CKPT} {FB}
save_runs()
import glob, re
for ck in sorted(glob.glob(f'{CKPT}/{MODEL}/grpo-s0/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1])):
    step = ck.rsplit('-', 1)[1]
    !python scripts/phase4_eval.py --label grpo-s0-step{step} --adapter {ck} --splits dev {FB}
!python scripts/phase4_eval.py --label grpo-s0 --adapter {CKPT}/{MODEL}/grpo-s0/final {FB}
save_runs()

## Step 4: GRPO seeds 1 and 2 (only after a "go")

In [ ]:
for s in (1, 2):
    !python scripts/phase4_train.py grpo --seed {s} --out {CKPT} {FB}
    !python scripts/phase4_eval.py --label grpo-s{s} --adapter {CKPT}/{MODEL}/grpo-s{s}/final {FB}
    save_runs()

In [ ]:
!cd runs && zip -q ../phase4_runs.zip *_phase4*.jsonl && cd .. && unzip -l phase4_runs.zip | tail -1
from google.colab import files; files.download('phase4_runs.zip')